# QSVM — nested CV with representation selection


**Expected runtime** (reference machine, all 100 outer folds, full grids):
ZZ ≈ 9 min, Z ≈ 9 min, classical ≈ 5 min, doubled for the group-aware re-run —
roughly 45 min end to end. This is only feasible because of the analytic
statevector path in §5; with `Statevector.from_instruction` per data point the
same run is on the order of a day.

## 1. Configuration and imports

In [28]:
import os
import json
import time
import hashlib
import itertools
import numpy as np
import pandas as pd
from scipy import stats
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial.distance import pdist, squareform

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, StratifiedGroupKFold
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (balanced_accuracy_score, accuracy_score,
                             matthews_corrcoef, confusion_matrix)
from sklearn.metrics.pairwise import rbf_kernel, linear_kernel, polynomial_kernel

import qiskit
from qiskit.circuit.library import zz_feature_map, z_feature_map
from qiskit.quantum_info import Statevector

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ---- cross-validation design (identical for every model, so runs are comparable)
N_SPLITS_OUTER  = 5
N_REPEATS_OUTER = 20      # -> 100 outer folds
N_SPLITS_INNER  = 4

# ---- hyperparameter grids, stated in full (Steve, section 8)
PARAM_GRID_QUANTUM = {
    'reps':      [1, 2],
    'bandwidth': [0.05, 0.1, 0.2, 0.5],
    'C':         [0.1, 1, 10],
}
CLASSICAL_GRIDS = {
    'linear': {'C': [0.1, 1, 10]},
    'rbf':    {'C': [0.1, 1, 10], 'gamma': ['scale', 0.01, 0.1, 1]},
    'poly':   {'C': [0.1, 1, 10], 'degree': [2, 3], 'gamma': ['scale', 0.1, 1],
               'coef0': [0.0, 1.0]},
}
INNER_SELECTION_METRIC = 'balanced_accuracy'
ENTANGLEMENT = 'linear'

OUTDIR = 'results_nested'
os.makedirs(OUTDIR, exist_ok=True)

print('qiskit', qiskit.__version__, '| seed', RANDOM_SEED)
print(f'outer {N_SPLITS_OUTER}x{N_REPEATS_OUTER} = {N_SPLITS_OUTER*N_REPEATS_OUTER} folds, '
      f'inner {N_SPLITS_INNER}, selection metric = {INNER_SELECTION_METRIC}')

qiskit 1.4.5 | seed 42
outer 5x20 = 100 folds, inner 4, selection metric = balanced_accuracy


## 2. Labels, sequences, and the representation catalogue



In [29]:
REPRESENTATIONS = {
    'mean_z':               'zscale_dataset.csv',          # 5  mean z-scale (order 1)
    # 'moments_order2':       'moments_order2_z5.csv',       # 10 mean + std
    # 'moments_order3':       'moments_order3_z5.csv',       # 15 mean + std + skew
    # 'pca4_mean_std':        'PCA_scaled_dataframe.csv',    # 8  PCA-4 pooled mean/std
    'z1_positional':        'zscaled_z1_only.csv',         # 9  first z-value, per position
    'pc1_positional_87':    'PC1_data_frame_87.csv',       # 9  PCA-1D (87 aa fit), per position
    'hopp_woods':           'hopp_woods.csv',              # 9  hydrophilicity
    'kyte_doolittle':       'kyte_doolittle.csv',          # 9  hydrophobicity
    'fauchere_pliska':      'fauchere_pliska.csv',         # 9  hydrophobicity
    'euclid_centroid':      'euclid_centroid.csv',         # 9  Euclidean centroid distance
    'mahalanobis_centroid': 'mahalanobis_centroid.csv',    # 9  Mahalanobis centroid distance
}

LABEL_LIKE_NAMES = {'y', 'label', 'class', 'target', 'binding_energy', 'energy'}


def load_representation(csv_path, sequences, y):
    data = pd.read_csv(csv_path)

    # layout B: per-amino-acid scale table -> expand to one feature per position
    if 'aa1' in data.columns:
        value_cols = [c for c in data.columns if c not in ('aa1', 'aa3')]
        if len(value_cols) != 1:
            raise ValueError(f'{csv_path}: expected exactly one value column, got {value_cols}')
        lookup = dict(zip(data['aa1'], data[value_cols[0]]))
        missing = set(''.join(sequences)) - set(lookup)
        if missing:
            raise ValueError(f'{csv_path}: no scale value for residues {missing}')
        X = np.array([[lookup[r] for r in seq] for seq in sequences], dtype=float)
        return X, [], []

    # layout A: peptide-level table
    dropped_non_numeric = [c for c in data.columns if not pd.api.types.is_numeric_dtype(data[c])]
    data = data.drop(columns=dropped_non_numeric)

    dropped_label = []
    for c in list(data.columns):
        col = data[c].values
        by_name = c.strip().lower() in LABEL_LIKE_NAMES
        by_value = set(np.unique(col)) <= {0, 1} and np.array_equal(col.astype(int), y)
        if by_name or by_value:
            dropped_label.append(c)
    data = data.drop(columns=dropped_label)
    return data.values.astype(float), dropped_non_numeric, dropped_label


def load_all_representations(sequences, y, catalogue=REPRESENTATIONS):
    reps, rows = {}, []
    for name, path in catalogue.items():
        X, non_num, lab = load_representation(path, sequences, y)
        assert X.shape[0] == len(y), f'{name}: {X.shape[0]} rows, expected {len(y)}'
        assert np.isfinite(X).all(), f'{name}: non-finite values present'
        reps[name] = X
        rows.append({'representation': name, 'source': path,
                     'n_features (= n_qubits)': X.shape[1],
                     'dropped_non_numeric': ', '.join(non_num) or '-',
                     'dropped_label_like': ', '.join(lab) or '-'})
    return reps, pd.DataFrame(rows)


_lab = pd.read_excel('Docking_high_low_energy_labels.xlsx', sheet_name='Results')
_lab = _lab.dropna(subset=['Sequence Epitope)']).reset_index(drop=True)
sequences = _lab['Sequence Epitope)'].astype(str).str.strip().tolist()
y = (_lab['Otsu Class theshold -77.8'] == 'Strong').astype(int).values

representations, rep_table = load_all_representations(sequences, y)
rep_table.to_csv(f'{OUTDIR}/representation_catalogue.csv', index=False)

print(f'{len(y)} peptides, class balance {np.bincount(y)}, '
      f'{len(set(sequences))} distinct sequences')
print()
display(rep_table)

80 peptides, class balance [42 38], 80 distinct sequences



/home/luispabloelchido/miniconda3/envs/QML/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,representation,source,n_features (= n_qubits),dropped_non_numeric,dropped_label_like
0,mean_z,zscale_dataset.csv,5,-,-
1,z1_positional,zscaled_z1_only.csv,9,Unnamed: 0,-
2,pc1_positional_87,PC1_data_frame_87.csv,9,-,-
3,hopp_woods,hopp_woods.csv,9,-,-
4,kyte_doolittle,kyte_doolittle.csv,9,-,-
5,fauchere_pliska,fauchere_pliska.csv,9,-,-
6,euclid_centroid,euclid_centroid.csv,9,-,-
7,mahalanobis_centroid,mahalanobis_centroid.csv,9,-,-


## 3. Sequence-redundancy groups



In [30]:
K_SHARED = 5
IDENTITY_THRESH = 7 / 9


def similarity_edges(sequences, k_shared=K_SHARED, identity_thresh=IDENTITY_THRESH):
    n, L = len(sequences), len(sequences[0])
    kmers = [{s[i:i + k_shared] for i in range(L - k_shared + 1)} for s in sequences]
    edges = []
    for i in range(n):
        for j in range(i + 1, n):
            shared = bool(kmers[i] & kmers[j])
            identity = sum(a == b for a, b in zip(sequences[i], sequences[j])) / L
            if shared or identity >= identity_thresh:
                edges.append((i, j))
    return edges


def build_groups_unionfind(n, edges):
    parent = list(range(n))

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for i, j in edges:
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[ri] = rj
    raw = [find(i) for i in range(n)]
    remap = {g: k for k, g in enumerate(sorted(set(raw)))}
    return np.array([remap[g] for g in raw])


n_pep = len(sequences)
edges = similarity_edges(sequences)
groups = build_groups_unionfind(n_pep, edges)

# --- independent verification that these are the connected components
adj = coo_matrix((np.ones(len(edges)),
                  (np.array([e[0] for e in edges]), np.array([e[1] for e in edges]))),
                 shape=(n_pep, n_pep))
n_cc, cc_labels = connected_components(adj, directed=False)


def same_partition(a, b):
    return len({(x, z) for x, z in zip(a, b)}) == len(set(a)) == len(set(b))


assert same_partition(groups, cc_labels), 'union-find groups are NOT the connected components'

sizes = pd.Series(groups).value_counts()
edge_set = set(edges)
non_clique = sum(
    1 for g in np.unique(groups)
    if (m := np.where(groups == g)[0]).size > 2
    and any((min(a, b), max(a, b)) not in edge_set
            for a, b in itertools.combinations(m, 2))
)

print(f'similarity edges          : {len(edges)}')
print(f'groups (union-find)       : {len(np.unique(groups))}')
print(f'groups (scipy components) : {n_cc}   <- must match, and the partitions are identical')
print(f'singleton groups          : {(sizes == 1).sum()}')
print(f'largest group size        : {sizes.max()}')
print(f'groups held together only by transitive closure: {non_clique}')
print('   (these contain at least one pair that is NOT directly similar --')
print('    with a non-transitive pairwise rule they would have been split apart)')

similarity edges          : 56
groups (union-find)       : 43
groups (scipy components) : 43   <- must match, and the partitions are identical
singleton groups          : 25
largest group size        : 7
groups held together only by transitive closure: 5
   (these contain at least one pair that is NOT directly similar --
    with a non-transitive pairwise rule they would have been split apart)


## 4. Pooled-feature duplicates (Steve §6, §7.6)



In [31]:
def duplicate_report(X, groups, y, near_tol=0.15):
    D = squareform(pdist(X))
    np.fill_diagonal(D, np.inf)
    exact = [(i, j) for i, j in zip(*np.where(D <= 1e-9)) if i < j]
    near = [(i, j) for i, j in zip(*np.where(D <= near_tol)) if i < j]
    split_across = [(i, j) for i, j in near if groups[i] != groups[j]]
    label_conflict = [(i, j) for i, j in near if y[i] != y[j]]
    return {'n_exact_duplicate_pairs': len(exact),
            'n_near_duplicate_pairs': len(near),
            'near_pairs_in_DIFFERENT_groups': len(split_across),
            'near_pairs_with_CONFLICTING_labels': len(label_conflict),
            'min_nn_distance': float(D.min()),
            'median_nn_distance': float(np.median(D.min(axis=1))),
            '_near': near, '_split': split_across}


dup_rows, detail = [], []
for name, X in representations.items():
    r = duplicate_report(X, groups, y)
    dup_rows.append({'representation': name, **{k: v for k, v in r.items() if not k.startswith('_')}})
    for i, j in r['_near']:
        detail.append({'representation': name, 'i': i, 'j': j,
                       'seq_i': sequences[i], 'seq_j': sequences[j],
                       'label_i': y[i], 'label_j': y[j],
                       'group_i': groups[i], 'group_j': groups[j],
                       'same_group': groups[i] == groups[j]})

dup_df = pd.DataFrame(dup_rows)
detail_df = pd.DataFrame(detail)
dup_df.to_csv(f'{OUTDIR}/pooled_feature_duplicates.csv', index=False)
detail_df.to_csv(f'{OUTDIR}/pooled_feature_duplicate_pairs.csv', index=False)

display(dup_df)
print()
if len(detail_df):
    print('near-duplicate pairs (distance <= 0.15):')
    display(detail_df)

n_bad = int(dup_df['near_pairs_in_DIFFERENT_groups'].sum())
if n_bad == 0:
    print('\nNo near-duplicate pair is split across groups: for this dataset the '
          'sequence-level grouping already keeps collapsed feature vectors together, '
          'so composition-space regrouping is not required.')
else:
    print(f'\n{n_bad} near-duplicate pair(s) fall in DIFFERENT groups -- the grouping '
          'should be redone in composition/feature space (Steve section 6).')

,representation,n_exact_duplicate_pairs,n_near_duplicate_pairs,near_pairs_in_DIFFERENT_groups,near_pairs_with_CONFLICTING_labels,min_nn_distance,median_nn_distance
0,mean_z,1,2,0,0,0.0000,0.5046
1,z1_positional,0,2,0,1,0.0564,4.4147
2,pc1_positional_87,0,2,0,1,0.0526,4.3960
3,hopp_woods,1,1,0,0,0.0000,2.7090
4,kyte_doolittle,0,0,0,0,0.3000,4.5659
5,fauchere_pliska,0,3,0,1,0.0200,1.5703
6,euclid_centroid,0,2,0,1,0.0128,1.8744
7,mahalanobis_centroid,0,4,0,0,0.0174,0.8791



near-duplicate pairs (distance <= 0.15):


,representation,i,j,seq_i,seq_j,label_i,label_j,group_i,group_j,same_group
0,mean_z,5,12,SSKTSVALG,SSKTSAVLG,0,0,6,6,True
1,mean_z,43,44,LPTHHTVRL,PTHHTVRLI,1,1,22,22,True
2,z1_positional,33,37,LVPGLKSLV,LVPGFKSLV,0,0,20,20,True
3,z1_positional,34,38,PGLKSLVLG,PGFKSLVLG,0,1,20,20,True
4,pc1_positional_87,33,37,LVPGLKSLV,LVPGFKSLV,0,0,20,20,True
5,pc1_positional_87,34,38,PGLKSLVLG,PGFKSLVLG,0,1,20,20,True
6,hopp_woods,63,71,TNRVALTMG,TNKVALTMG,0,0,39,39,True
7,fauchere_pliska,33,37,LVPGLKSLV,LVPGFKSLV,0,0,20,20,True
8,fauchere_pliska,34,38,PGLKSLVLG,PGFKSLVLG,0,1,20,20,True
9,fauchere_pliska,63,71,TNRVALTMG,TNKVALTMG,0,0,39,39,True



No near-duplicate pair is split across groups: for this dataset the sequence-level grouping already keeps collapsed feature vectors together, so composition-space regrouping is not required.


## 5. Exact fidelity kernel — analytic statevectors



In [32]:
def _fwht_inplace(a):
    """Unnormalised Walsh-Hadamard transform along axis 1 (H^(x)n up to a factor)."""
    N, dim = a.shape
    h = 1
    while h < dim:
        v = a.reshape(N, -1, 2, h)
        x, yy = v[:, :, 0, :], v[:, :, 1, :]
        t = x - yy
        x += yy
        yy[...] = t
        h *= 2
    return a


def statevectors(X, kind, reps, entanglement=ENTANGLEMENT):
    """Exact statevectors of qiskit's z_feature_map / zz_feature_map, vectorised over rows of X."""
    X = np.asarray(X, dtype=float)
    N, n = X.shape
    dim = 2 ** n
    if kind not in ('z', 'zz'):
        raise ValueError(kind)
    if kind == 'zz' and entanglement != 'linear':
        raise ValueError("only 'linear' entanglement is implemented")

    # single-qubit phase factors: exp(1j * 2 * x_i * b_i)
    single = np.empty((n, N, 2), dtype=complex)
    single[:, :, 0] = 1.0
    single[:, :, 1] = np.exp(1j * 2.0 * X.T)

    if kind == 'zz':
        # nearest-neighbour phase: exp(1j * 2 * (x_i - pi)(x_j - pi) * (b_i XOR b_j))
        pair = np.ones((n - 1, N, 2, 2), dtype=complex)
        for i in range(n - 1):
            e = np.exp(1j * 2.0 * (X[:, i] - np.pi) * (X[:, i + 1] - np.pi))
            pair[i, :, 0, 1] = e
            pair[i, :, 1, 0] = e

    # assemble the diagonal from qubit n-1 down to qubit 0 so the flat index is
    # sum_i b_i * 2**i, matching qiskit's little-endian convention
    acc = single[n - 1]
    for i in range(n - 2, -1, -1):
        acc = acc.reshape(N, -1, 2)
        new = acc[:, :, :, None] * single[i][:, None, None, :]
        if kind == 'zz':
            new = new * pair[i][:, None, :, :]
        acc = new.reshape(N, -1)
    diag = acc

    state = diag * (dim ** -0.5)              # H^(x)n on |0...0>, then the phase block
    for _ in range(reps - 1):
        state = _fwht_inplace(state) * (dim ** -0.5)
        state *= diag
    return state


def kernel_train_test(X_tr, X_te, kind, reps, entanglement=ENTANGLEMENT):
    """Returns (K_train, K_test_vs_train) from a single statevector pass."""
    sv = statevectors(np.vstack([X_tr, X_te]), kind, reps, entanglement)
    m = len(X_tr)
    return np.abs(sv[:m].conj() @ sv[:m].T) ** 2, np.abs(sv[m:].conj() @ sv[:m].T) ** 2

In [33]:
# ---- validation against qiskit's circuit simulation
rng = np.random.default_rng(0)
worst, rows = 0.0, []
for n in [2, 3, 5, 8, 9, 10]:
    for kind in ['z', 'zz']:
        for reps in [1, 2, 3]:
            Xv = rng.uniform(-np.pi, np.pi, size=(5, n))
            fm = (zz_feature_map(n, reps=reps, entanglement='linear') if kind == 'zz'
                  else z_feature_map(n, reps=reps))
            ref = np.array([Statevector.from_instruction(fm.assign_parameters(x)).data for x in Xv])
            err = np.abs(ref - statevectors(Xv, kind, reps)).max()
            K_ref = np.abs(ref.conj() @ ref.T) ** 2
            K_new = np.abs(statevectors(Xv, kind, reps).conj() @ statevectors(Xv, kind, reps).T) ** 2
            worst = max(worst, err)
            rows.append({'n_qubits': n, 'map': kind, 'reps': reps,
                         'max|dStatevector|': err, 'max|dKernel|': np.abs(K_ref - K_new).max()})
val_df = pd.DataFrame(rows)
val_df.to_csv(f'{OUTDIR}/statevector_validation.csv', index=False)
assert worst < 1e-10, 'analytic statevectors disagree with qiskit'
print(f'analytic path matches qiskit on all {len(rows)} configurations; '
      f'worst deviation {worst:.2e}')
display(val_df.groupby(['map', 'reps'])[['max|dStatevector|', 'max|dKernel|']].max())

analytic path matches qiskit on all 36 configurations; worst deviation 1.01e-15


max|dStatevector|  max|dKernel|
map reps                                 
z   1                0.0000        0.0000
    2                0.0000        0.0000
    3                0.0000        0.0000
zz  1                0.0000        0.0000
    2                0.0000        0.0000
    3                0.0000        0.0000

## 6. Outer/inner splits, with provenance



In [34]:
def repeated_stratified_group_kfold(y, groups, n_splits, n_repeats, random_state=RANDOM_SEED):
    """StratifiedGroupKFold is not natively repeatable; re-shuffle per repetition."""
    splits = []
    for r in range(n_repeats):
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state + r)
        splits.extend(sgkf.split(np.zeros(len(y)), y, groups))
    return splits


def make_outer_splits(y, groups=None, n_splits=N_SPLITS_OUTER,
                      n_repeats=N_REPEATS_OUTER, random_state=RANDOM_SEED):
    if groups is None:
        return list(RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats,
                                            random_state=random_state).split(np.zeros(len(y)), y))
    return repeated_stratified_group_kfold(y, groups, n_splits, n_repeats, random_state)


def split_fingerprints(splits):
    out = []
    for tr, te in splits:
        payload = np.concatenate([np.sort(tr), np.array([-1]), np.sort(te)]).astype(np.int64)
        out.append(hashlib.sha1(payload.tobytes()).hexdigest())
    return out


def audit_splits(splits, groups, name):
    fps = split_fingerprints(splits)
    leaky = sum(1 for tr, te in splits if set(groups[tr]) & set(groups[te]))
    appearances = np.bincount(np.concatenate([te for _, te in splits]), minlength=len(y))
    pd.DataFrame({'fold': range(len(splits)), 'sha1': fps}).to_csv(
        f'{OUTDIR}/split_fingerprints_{name}.csv', index=False)
    return {'scheme': name, 'n_outer_folds': len(splits), 'unique_partitions': len(set(fps)),
            'folds_with_group_leakage': leaky,
            'test_appearances_min': int(appearances.min()),
            'test_appearances_max': int(appearances.max()),
            'mean_n_train': float(np.mean([len(tr) for tr, _ in splits])),
            'mean_n_test': float(np.mean([len(te) for _, te in splits]))}


splits_random = make_outer_splits(y, None)
splits_grouped = make_outer_splits(y, groups)
audit = pd.DataFrame([audit_splits(splits_random, groups, 'random'),
                      audit_splits(splits_grouped, groups, 'grouped')])
audit.to_csv(f'{OUTDIR}/split_audit.csv', index=False)
display(audit)

assert audit.loc[1, 'folds_with_group_leakage'] == 0, 'group leakage in the group-aware splits'
print('\nEvery repetition produced a distinct partition, and no group spans train/test')
print('in the group-aware scheme. The random-split scheme leaks groups in')
print(f"{audit.loc[0, 'folds_with_group_leakage']} of {len(splits_random)} folds, which is exactly")
print('why the group-aware numbers are the ones to report as primary.')

,scheme,n_outer_folds,unique_partitions,folds_with_group_leakage,test_appearances_min,test_appearances_max,mean_n_train,mean_n_test
0,random,100,100,100,20,20,64.0000,16.0000
1,grouped,100,100,0,20,20,64.0000,16.0000



Every repetition produced a distinct partition, and no group spans train/test
in the group-aware scheme. The random-split scheme leaks groups in
100 of 100 folds, which is exactly
why the group-aware numbers are the ones to report as primary.


## 7. Nested CV with the representation inside the inner loop



In [35]:
def fit_scale(X_train, X_test, bandwidth=1.0):
    """A new scaler per fold, fit ONLY on that fold's training part."""
    scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
    scaler.fit(X_train)
    return scaler.transform(X_train) * bandwidth, scaler.transform(X_test) * bandwidth


def kernel_diagnostics(K, y_pm1):
    """KTA, CKA and effective rank, computed on the OUTER TRAINING Gram matrix only."""
    m = len(y_pm1)
    S = y_pm1 @ K @ y_pm1 - np.trace(K)
    Q = np.sum(K ** 2) - np.sum(np.diag(K) ** 2)
    kta = (m + S) / (m * np.sqrt(m + Q))
    H = np.eye(m) - np.ones((m, m)) / m
    Kc = H @ K @ H
    yyTc = H @ np.outer(y_pm1, y_pm1).astype(float) @ H
    cka = np.sum(Kc * yyTc) / (np.linalg.norm(Kc) * np.linalg.norm(yyTc) + 1e-15)
    eig = np.clip(np.linalg.eigvalsh((K + K.T) / 2), 0, None)
    eff_rank = (eig.sum() ** 2) / (np.sum(eig ** 2) + 1e-15)
    return kta, cka, eff_rank


def _param_combinations(grid):
    keys = list(grid)
    return [dict(zip(keys, vals)) for vals in itertools.product(*(grid[k] for k in keys))]


def _resolve_gamma(gamma, Xs):
    if gamma == 'scale':
        return 1.0 / (Xs.shape[1] * Xs.var()) if Xs.var() > 0 else 1.0
    return gamma


def _fit_svc(kernel_name, Xtr, ytr, params):
    gamma = _resolve_gamma(params.get('gamma', 'scale'), Xtr)
    if kernel_name == 'linear':
        clf = SVC(kernel='linear', C=params['C'])
    elif kernel_name == 'rbf':
        clf = SVC(kernel='rbf', C=params['C'], gamma=gamma)
    else:
        clf = SVC(kernel='poly', C=params['C'], degree=params['degree'],
                  gamma=gamma, coef0=params['coef0'])
    return clf.fit(Xtr, ytr)


def _build_gram(kernel_name, Xa, Xb, params, Xtr):
    gamma = _resolve_gamma(params.get('gamma', 'scale'), Xtr)
    if kernel_name == 'linear':
        return linear_kernel(Xa, Xb)
    if kernel_name == 'rbf':
        return rbf_kernel(Xa, Xb, gamma=gamma)
    return polynomial_kernel(Xa, Xb, degree=params['degree'], gamma=gamma, coef0=params['coef0'])


def _inner_splits(y_tr, groups_tr, fold_i):
    if groups_tr is None:
        return list(StratifiedKFold(n_splits=N_SPLITS_INNER, shuffle=True,
                                    random_state=RANDOM_SEED + fold_i)
                    .split(np.zeros(len(y_tr)), y_tr))
    return list(StratifiedGroupKFold(n_splits=N_SPLITS_INNER, shuffle=True,
                                     random_state=RANDOM_SEED + fold_i)
                .split(np.zeros(len(y_tr)), y_tr, groups_tr))

In [36]:
def nested_cv_quantum(reps_data, y, kind, groups=None, param_grid=PARAM_GRID_QUANTUM,
                      entanglement=ENTANGLEMENT, verbose=True):
    outer = make_outer_splits(y, groups)
    rows, predictions = [], {}
    t0 = time.time()

    for fold_i, (tr_idx, te_idx) in enumerate(outer):
        y_tr, y_te = y[tr_idx], y[te_idx]
        inner = _inner_splits(y_tr, None if groups is None else groups[tr_idx], fold_i)

        inner_scores = {}
        for rep_name, X_all in reps_data.items():
            X_tr_outer = X_all[tr_idx]
            for in_tr, in_val in inner:
                yi_tr, yi_val = y_tr[in_tr], y_tr[in_val]
                for bw in param_grid['bandwidth']:
                    Xi_tr, Xi_val = fit_scale(X_tr_outer[in_tr], X_tr_outer[in_val], bandwidth=bw)
                    for reps in param_grid['reps']:
                        K_tr, K_val = kernel_train_test(Xi_tr, Xi_val, kind, reps, entanglement)
                        for C in param_grid['C']:
                            clf = SVC(kernel='precomputed', C=C).fit(K_tr, yi_tr)
                            s = balanced_accuracy_score(yi_val, clf.predict(K_val))
                            inner_scores.setdefault((rep_name, reps, bw, C), []).append(s)

        best = max(inner_scores, key=lambda k: np.mean(inner_scores[k]))
        best_rep, best_reps, best_bw, best_C = best

        X_best = reps_data[best_rep]
        Xtr, Xte = fit_scale(X_best[tr_idx], X_best[te_idx], bandwidth=best_bw)
        K_tr, K_te = kernel_train_test(Xtr, Xte, kind, best_reps, entanglement)
        clf = SVC(kernel='precomputed', C=best_C).fit(K_tr, y_tr)
        y_pred = clf.predict(K_te)
        kta, cka, eff = kernel_diagnostics(K_tr, np.where(y_tr == 1, 1, -1))

        rows.append({'fold': fold_i, 'best_representation': best_rep, 'best_reps': best_reps,
                     'best_bandwidth': best_bw, 'best_C': best_C,
                     'inner_score': float(np.mean(inner_scores[best])),
                     'accuracy': accuracy_score(y_te, y_pred),
                     'balanced_accuracy': balanced_accuracy_score(y_te, y_pred),
                     'mcc': matthews_corrcoef(y_te, y_pred) if len(set(y_pred)) > 1 else 0.0,
                     'kta': kta, 'cka': cka, 'effective_rank': eff,
                     'n_train': len(tr_idx), 'n_test': len(te_idx)})
        predictions[fold_i] = {'test_idx': te_idx.tolist(), 'y_true': y_te.tolist(),
                               'y_pred': y_pred.tolist()}
        if verbose and fold_i % 20 == 0:
            print(f'   [{kind}{"" if groups is None else "/grouped"}] fold {fold_i}/{len(outer)}'
                  f'  {time.time()-t0:5.0f}s  -> {best_rep}')
    return pd.DataFrame(rows), predictions


def nested_cv_classical(reps_data, y, kernel_name, groups=None, verbose=True):
    combos = _param_combinations(CLASSICAL_GRIDS[kernel_name])
    outer = make_outer_splits(y, groups)
    rows, predictions = [], {}
    t0 = time.time()

    for fold_i, (tr_idx, te_idx) in enumerate(outer):
        y_tr, y_te = y[tr_idx], y[te_idx]
        inner = _inner_splits(y_tr, None if groups is None else groups[tr_idx], fold_i)

        inner_scores = {}
        for rep_name, X_all in reps_data.items():
            X_tr_outer = X_all[tr_idx]
            for in_tr, in_val in inner:
                yi_tr, yi_val = y_tr[in_tr], y_tr[in_val]
                Xi_tr, Xi_val = fit_scale(X_tr_outer[in_tr], X_tr_outer[in_val])
                for ci, params in enumerate(combos):
                    clf = _fit_svc(kernel_name, Xi_tr, yi_tr, params)
                    s = balanced_accuracy_score(yi_val, clf.predict(Xi_val))
                    inner_scores.setdefault((rep_name, ci), []).append(s)

        best = max(inner_scores, key=lambda k: np.mean(inner_scores[k]))
        best_rep, best_ci = best
        best_params = combos[best_ci]

        X_best = reps_data[best_rep]
        Xtr, Xte = fit_scale(X_best[tr_idx], X_best[te_idx])
        clf = _fit_svc(kernel_name, Xtr, y_tr, best_params)
        y_pred = clf.predict(Xte)
        K_tr = _build_gram(kernel_name, Xtr, Xtr, best_params, Xtr)
        kta, cka, eff = kernel_diagnostics(K_tr, np.where(y_tr == 1, 1, -1))

        row = {'fold': fold_i, 'best_representation': best_rep,
               'inner_score': float(np.mean(inner_scores[best])),
               'accuracy': accuracy_score(y_te, y_pred),
               'balanced_accuracy': balanced_accuracy_score(y_te, y_pred),
               'mcc': matthews_corrcoef(y_te, y_pred) if len(set(y_pred)) > 1 else 0.0,
               'kta': kta, 'cka': cka, 'effective_rank': eff,
               'n_train': len(tr_idx), 'n_test': len(te_idx)}
        row.update({f'best_{k}': v for k, v in best_params.items()})
        rows.append(row)
        predictions[fold_i] = {'test_idx': te_idx.tolist(), 'y_true': y_te.tolist(),
                               'y_pred': y_pred.tolist()}
        if verbose and fold_i % 50 == 0:
            print(f'   [{kernel_name}{"" if groups is None else "/grouped"}] fold {fold_i}/'
                  f'{len(outer)}  {time.time()-t0:5.0f}s  -> {best_rep}')
    return pd.DataFrame(rows), predictions


def majority_baseline(y, groups=None):
    outer = make_outer_splits(y, groups)
    rows, predictions = [], {}
    for fold_i, (tr_idx, te_idx) in enumerate(outer):
        clf = DummyClassifier(strategy='most_frequent').fit(np.zeros((len(tr_idx), 1)), y[tr_idx])
        y_pred = clf.predict(np.zeros((len(te_idx), 1)))
        rows.append({'fold': fold_i, 'best_representation': '-',
                     'accuracy': accuracy_score(y[te_idx], y_pred),
                     'balanced_accuracy': balanced_accuracy_score(y[te_idx], y_pred),
                     'mcc': 0.0, 'n_train': len(tr_idx), 'n_test': len(te_idx)})
        predictions[fold_i] = {'test_idx': te_idx.tolist(), 'y_true': y[te_idx].tolist(),
                               'y_pred': y_pred.tolist()}
    return pd.DataFrame(rows), predictions

## 8. Run — random splits

Kept only as the comparison point that shows how much sequence redundancy
inflates the estimate. The group-aware run in §9 is the one to report.

In [37]:
MODELS = ['ZZFeatureMap', 'ZFeatureMap', 'Classical linear', 'Classical RBF', 'Classical poly']

results, preds = {}, {}
t_start = time.time()

for label, kind in [('ZZFeatureMap', 'zz'), ('ZFeatureMap', 'z')]:
    t0 = time.time()
    results[label], preds[label] = nested_cv_quantum(representations, y, kind)
    print(f'{label}: {time.time()-t0:.0f}s')

for kn, label in [('linear', 'Classical linear'), ('rbf', 'Classical RBF'), ('poly', 'Classical poly')]:
    t0 = time.time()
    results[label], preds[label] = nested_cv_classical(representations, y, kn)
    print(f'{label}: {time.time()-t0:.0f}s')

results['Majority baseline'], preds['Majority baseline'] = majority_baseline(y)

for label in results:
    results[label].to_csv(f'{OUTDIR}/folds_random_{label.replace(" ", "_")}.csv', index=False)
json.dump({k: v for k, v in preds.items()}, open(f'{OUTDIR}/predictions_random.json', 'w'))
print(f'\ntotal {time.time()-t_start:.0f}s')

   [zz] fold 0/100      2s  -> mean_z
   [zz] fold 20/100     32s  -> mean_z
   [zz] fold 40/100     61s  -> mean_z
   [zz] fold 60/100     90s  -> mean_z
   [zz] fold 80/100    119s  -> mean_z
ZZFeatureMap: 147s
   [z] fold 0/100      1s  -> mean_z
   [z] fold 20/100     29s  -> mean_z
   [z] fold 40/100     56s  -> mean_z
   [z] fold 60/100     84s  -> mean_z
   [z] fold 80/100    111s  -> mean_z
ZFeatureMap: 137s
   [linear] fold 0/100      0s  -> mean_z
   [linear] fold 50/100     15s  -> mean_z
Classical linear: 29s
   [rbf] fold 0/100      1s  -> mean_z
   [rbf] fold 50/100     26s  -> mean_z
Classical RBF: 52s
   [poly] fold 0/100      2s  -> mean_z
   [poly] fold 50/100     83s  -> mean_z
Classical poly: 163s

total 528s


## 9. Run — group-aware splits (primary)

In [38]:
results_grp, preds_grp = {}, {}
t_start = time.time()

for label, kind in [('ZZFeatureMap', 'zz'), ('ZFeatureMap', 'z')]:
    t0 = time.time()
    results_grp[label], preds_grp[label] = nested_cv_quantum(representations, y, kind, groups=groups)
    print(f'{label} (group-aware): {time.time()-t0:.0f}s')

for kn, label in [('linear', 'Classical linear'), ('rbf', 'Classical RBF'), ('poly', 'Classical poly')]:
    t0 = time.time()
    results_grp[label], preds_grp[label] = nested_cv_classical(representations, y, kn, groups=groups)
    print(f'{label} (group-aware): {time.time()-t0:.0f}s')

results_grp['Majority baseline'], preds_grp['Majority baseline'] = majority_baseline(y, groups=groups)

for label in results_grp:
    results_grp[label].to_csv(f'{OUTDIR}/folds_grouped_{label.replace(" ", "_")}.csv', index=False)
json.dump({k: v for k, v in preds_grp.items()}, open(f'{OUTDIR}/predictions_grouped.json', 'w'))
print(f'\ntotal {time.time()-t_start:.0f}s')

   [zz/grouped] fold 0/100      2s  -> mean_z
   [zz/grouped] fold 20/100     31s  -> mean_z
   [zz/grouped] fold 40/100     61s  -> mean_z
   [zz/grouped] fold 60/100     90s  -> mean_z
   [zz/grouped] fold 80/100    119s  -> mean_z
ZZFeatureMap (group-aware): 148s
   [z/grouped] fold 0/100      1s  -> mean_z
   [z/grouped] fold 20/100     31s  -> mean_z
   [z/grouped] fold 40/100     60s  -> mean_z
   [z/grouped] fold 60/100     88s  -> mean_z
   [z/grouped] fold 80/100    115s  -> mean_z
ZFeatureMap (group-aware): 142s
   [linear/grouped] fold 0/100      0s  -> mean_z
   [linear/grouped] fold 50/100     16s  -> mean_z
Classical linear (group-aware): 33s
   [rbf/grouped] fold 0/100      1s  -> mean_z
   [rbf/grouped] fold 50/100     27s  -> mean_z
Classical RBF (group-aware): 55s
   [poly/grouped] fold 0/100      2s  -> mean_z
   [poly/grouped] fold 50/100     88s  -> mean_z
Classical poly (group-aware): 172s

total 549s


## 10. What the procedure actually selected

Because the representation is chosen inside each outer fold, different folds may
choose differently. **That spread is a result, not a nuisance**: it measures how
stable the "winning representation" claim really is. If one representation is
picked in nearly every fold, the original screen was reading a real signal; if
the choice is scattered, the apparent winner in the earlier note was largely a
selection artefact — which is precisely the possibility Steve raised.

In [39]:
def selection_table(res_dict):
    rows = []
    for label, res in res_dict.items():
        if 'best_representation' not in res or set(res['best_representation']) == {'-'}:
            continue
        vc = res['best_representation'].value_counts()
        rows.append({'model': label, 'modal_representation': vc.index[0],
                     'modal_share': f'{vc.iloc[0] / len(res):.0%}',
                     'n_distinct_representations_used': res['best_representation'].nunique(),
                     **{f'n_{k}': int(v) for k, v in vc.items()}})
    return pd.DataFrame(rows).set_index('model').fillna(0)


sel_random = selection_table(results)
sel_grouped = selection_table(results_grp)
sel_random.to_csv(f'{OUTDIR}/representation_selection_random.csv')
sel_grouped.to_csv(f'{OUTDIR}/representation_selection_grouped.csv')

print('Representation chosen by the inner loop -- RANDOM splits')
display(sel_random)
print('\nRepresentation chosen by the inner loop -- GROUP-AWARE splits')
display(sel_grouped)

print('\nSelected quantum hyperparameters (group-aware):')
for label in ['ZZFeatureMap', 'ZFeatureMap']:
    r = results_grp[label]
    print(f'  {label}: bandwidth {r["best_bandwidth"].value_counts().to_dict()}, '
          f'reps {r["best_reps"].value_counts().to_dict()}, '
          f'C {r["best_C"].value_counts().to_dict()}')

Representation chosen by the inner loop -- RANDOM splits


,modal_representation,modal_share,n_distinct_representations_used,n_mean_z,n_kyte_doolittle
model,,,,,
ZZFeatureMap,mean_z,100%,1,100,0.0000
ZFeatureMap,mean_z,100%,1,100,0.0000
Classical linear,mean_z,99%,2,99,1.0000
Classical RBF,mean_z,100%,1,100,0.0000
Classical poly,mean_z,100%,1,100,0.0000



Representation chosen by the inner loop -- GROUP-AWARE splits


,modal_representation,modal_share,n_distinct_representations_used,n_mean_z,n_hopp_woods,n_euclid_centroid
model,,,,,,
ZZFeatureMap,mean_z,100%,1,100,0.0000,0.0000
ZFeatureMap,mean_z,99%,2,99,1.0000,0.0000
Classical linear,mean_z,99%,2,99,0.0000,1.0000
Classical RBF,mean_z,100%,1,100,0.0000,0.0000
Classical poly,mean_z,100%,1,100,0.0000,0.0000



Selected quantum hyperparameters (group-aware):
  ZZFeatureMap: bandwidth {0.05: 70, 0.1: 30}, reps {1: 62, 2: 38}, C {10: 76, 1: 24}
  ZFeatureMap: bandwidth {0.2: 57, 0.1: 35, 0.05: 8}, reps {1: 95, 2: 5}, C {10: 73, 1: 27}


## 11. Performance, and paired uncertainty done three ways

The `+/-` in the previous tables were standard errors over 100 outer folds
treated as independent. They are not: repeated folds share peptides and share
training sets, so those intervals are too narrow. Three intervals are reported
side by side so the difference is visible rather than argued:

1. **Fold bootstrap** — resamples outer folds. This is the old, optimistic
   interval, kept only as a reference point.
2. **Nadeau–Bengio corrected interval** — inflates the variance by
   `1/J + n_test/n_train` to account for the overlap between the training sets of
   repeated folds (reference [6]).
3. **Group bootstrap** — resamples **sequence groups**, which is the natural
   independent unit for the group-aware analysis. All out-of-fold predictions are
   pooled per peptide, groups are resampled with replacement, and balanced
   accuracy is recomputed for both models on the same resampled set, giving a
   genuinely paired difference.

Every comparison is paired on identical outer splits — never a comparison of two
separate error bars.

In [40]:
def per_fold_metrics(predictions):
    accs, baccs, mccs, sens, specs = [], [], [], [], []
    yt_all, yp_all = [], []
    for fold_i in sorted(predictions, key=int):
        yt = np.asarray(predictions[fold_i]['y_true'])
        yp = np.asarray(predictions[fold_i]['y_pred'])
        yt_all += yt.tolist(); yp_all += yp.tolist()
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        sens.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)
        accs.append(accuracy_score(yt, yp))
        baccs.append(balanced_accuracy_score(yt, yp))
        mccs.append(matthews_corrcoef(yt, yp) if len(set(yp)) > 1 else 0.0)
    return {'accuracy': np.array(accs), 'balanced_accuracy': np.array(baccs),
            'mcc': np.array(mccs), 'sensitivity': np.array(sens),
            'specificity': np.array(specs),
            'aggregate_confusion_matrix': confusion_matrix(yt_all, yp_all, labels=[0, 1])}


def bootstrap_paired_ci_folds(diff, n_boot=10000, seed=RANDOM_SEED):
    """Resampling unit = outer fold. Ignores that repeated folds share peptides."""
    rng = np.random.default_rng(seed)
    boots = rng.choice(diff, size=(n_boot, len(diff)), replace=True).mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(diff.mean()), float(lo), float(hi)


def nadeau_bengio_ci(diff, n_train, n_test, alpha=0.05):
    """Corrected resampled t-test, Nadeau & Bengio (2003)."""
    diff = np.asarray(diff, dtype=float)
    J = len(diff)
    var = diff.var(ddof=1)
    se = np.sqrt(var * (1.0 / J + n_test / n_train))
    t = stats.t.ppf(1 - alpha / 2, df=J - 1)
    mean = diff.mean()
    t_stat = mean / se if se > 0 else np.nan
    p = 2 * (1 - stats.t.cdf(abs(t_stat), df=J - 1)) if np.isfinite(t_stat) else np.nan
    return float(mean), float(mean - t * se), float(mean + t * se), float(p)


def _peptide_tallies(predictions, n_samples):
    n_pred = np.zeros(n_samples)
    n_correct = np.zeros(n_samples)
    for fold_i in sorted(predictions, key=int):
        idx = np.asarray(predictions[fold_i]['test_idx'])
        yt = np.asarray(predictions[fold_i]['y_true'])
        yp = np.asarray(predictions[fold_i]['y_pred'])
        n_pred[idx] += 1
        n_correct[idx] += (yt == yp)
    return n_pred, n_correct


def bootstrap_group_ci(pred_a, pred_b, groups, y, n_boot=10000, seed=RANDOM_SEED):
    """Paired bootstrap whose resampling unit is the SEQUENCE GROUP."""
    n = len(y)
    na, ca = _peptide_tallies(pred_a, n)
    nb, cb = _peptide_tallies(pred_b, n)
    uniq = np.unique(groups)
    members = [np.where(groups == g)[0] for g in uniq]
    pos = (y == 1)

    def bacc(n_pred, n_corr, idx):
        p, q = idx[pos[idx]], idx[~pos[idx]]
        tpr = n_corr[p].sum() / n_pred[p].sum() if n_pred[p].sum() > 0 else np.nan
        tnr = n_corr[q].sum() / n_pred[q].sum() if n_pred[q].sum() > 0 else np.nan
        return 0.5 * (tpr + tnr)

    rng = np.random.default_rng(seed)
    all_idx = np.arange(n)
    observed = bacc(na, ca, all_idx) - bacc(nb, cb, all_idx)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(uniq), size=len(uniq))])
        boots[b] = bacc(na, ca, idx) - bacc(nb, cb, idx)
    boots = boots[np.isfinite(boots)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(observed), float(lo), float(hi)

In [41]:
def summary_table(res_dict, pred_dict):
    rows = []
    for label, res in res_dict.items():
        m = per_fold_metrics(pred_dict[label])
        row = {'model': label, 'n_outer_folds': len(m['accuracy'])}
        for key in ['accuracy', 'balanced_accuracy', 'mcc', 'sensitivity', 'specificity']:
            row[f'{key}_mean'] = np.nanmean(m[key])
            row[f'{key}_se_naive'] = np.nanstd(m[key], ddof=1) / np.sqrt(np.sum(~np.isnan(m[key])))
        for key in ['kta', 'cka', 'effective_rank']:
            row[f'{key}_mean'] = res[key].mean() if key in res else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index('model')


pd.set_option('display.float_format', lambda v: f'{v:.4f}')
summary_random = summary_table(results, preds)
summary_grouped = summary_table(results_grp, preds_grp)
summary_random.to_csv(f'{OUTDIR}/summary_random.csv')
summary_grouped.to_csv(f'{OUTDIR}/summary_grouped.csv')

cols = ['accuracy_mean', 'balanced_accuracy_mean', 'balanced_accuracy_se_naive',
        'mcc_mean', 'sensitivity_mean', 'specificity_mean']
print('RANDOM splits (comparison point only)')
display(summary_random[cols])
print('\nGROUP-AWARE splits (primary)')
display(summary_grouped[cols])
print('\nKernel diagnostics, group-aware (train-fold Gram matrices only)')
display(summary_grouped[['kta_mean', 'cka_mean', 'effective_rank_mean']])

drop = pd.DataFrame({
    'balanced_accuracy_random': summary_random['balanced_accuracy_mean'],
    'balanced_accuracy_group_aware': summary_grouped['balanced_accuracy_mean'],
})
drop['drop'] = drop['balanced_accuracy_random'] - drop['balanced_accuracy_group_aware']
drop.to_csv(f'{OUTDIR}/random_vs_grouped.csv')
print('\nHow optimistic were the random splits?')
display(drop)

RANDOM splits (comparison point only)


,accuracy_mean,balanced_accuracy_mean,balanced_accuracy_se_naive,mcc_mean,sensitivity_mean,specificity_mean
model,,,,,,
ZZFeatureMap,0.8063,0.8073,0.0094,0.6248,0.8068,0.8078
ZFeatureMap,0.8313,0.8319,0.0090,0.6739,0.8332,0.8306
Classical linear,0.8325,0.8330,0.0095,0.6767,0.8246,0.8414
Classical RBF,0.8325,0.8333,0.0091,0.6767,0.8370,0.8296
Classical poly,0.8444,0.8449,0.0091,0.6987,0.8407,0.8490
Majority baseline,0.5250,0.5000,0.0000,0.0000,0.0000,1.0000



GROUP-AWARE splits (primary)


,accuracy_mean,balanced_accuracy_mean,balanced_accuracy_se_naive,mcc_mean,sensitivity_mean,specificity_mean
model,,,,,,
ZZFeatureMap,0.7835,0.7818,0.0119,0.5643,0.7443,0.8193
ZFeatureMap,0.8291,0.8270,0.0116,0.6512,0.8083,0.8456
Classical linear,0.8356,0.8293,0.0113,0.6586,0.8228,0.8358
Classical RBF,0.8099,0.8050,0.0115,0.6124,0.7695,0.8405
Classical poly,0.8296,0.8295,0.0113,0.6572,0.8182,0.8408
Majority baseline,0.4437,0.5000,0.0000,0.0000,0.1900,0.8100



Kernel diagnostics, group-aware (train-fold Gram matrices only)


,kta_mean,cka_mean,effective_rank_mean
model,,,
ZZFeatureMap,0.1461,0.2620,5.9111
ZFeatureMap,0.0802,0.3257,2.3811
Classical linear,0.3090,0.3337,3.8061
Classical RBF,0.0961,0.3145,4.0461
Classical poly,0.1963,0.2900,6.6067
Majority baseline,NaN,NaN,NaN



How optimistic were the random splits?


,balanced_accuracy_random,balanced_accuracy_group_aware,drop
model,,,
ZZFeatureMap,0.8073,0.7818,0.0255
ZFeatureMap,0.8319,0.8270,0.0049
Classical linear,0.8330,0.8293,0.0037
Classical RBF,0.8333,0.8050,0.0283
Classical poly,0.8449,0.8295,0.0154
Majority baseline,0.5000,0.5000,0.0000


In [42]:
COMPARISONS = [
    ('ZZFeatureMap', 'ZFeatureMap'),            # entanglement
    ('ZZFeatureMap', 'Classical RBF'),
    ('ZZFeatureMap', 'Classical linear'),
    ('ZZFeatureMap', 'Classical poly'),
    ('ZFeatureMap',  'Classical RBF'),
    ('ZFeatureMap',  'Classical linear'),
    ('ZFeatureMap',  'Classical poly'),
    ('ZZFeatureMap', 'Majority baseline'),
]


def comparison_table(res_dict, pred_dict, groups, y, scheme):
    metrics = {k: per_fold_metrics(v) for k, v in pred_dict.items()}
    rows = []
    for a, b in COMPARISONS:
        d = metrics[a]['balanced_accuracy'] - metrics[b]['balanced_accuracy']
        n_tr = res_dict[a]['n_train'].mean()
        n_te = res_dict[a]['n_test'].mean()
        m_f, lo_f, hi_f = bootstrap_paired_ci_folds(d)
        m_n, lo_n, hi_n, p_n = nadeau_bengio_ci(d, n_tr, n_te)
        m_g, lo_g, hi_g = bootstrap_group_ci(pred_dict[a], pred_dict[b], groups, y)
        rows.append({
            'comparison': f'{a} - {b}', 'scheme': scheme, 'mean_diff': m_f,
            'fold_boot_lo': lo_f, 'fold_boot_hi': hi_f,
            'fold_boot_excl_0': (lo_f > 0) or (hi_f < 0),
            'nadeau_bengio_lo': lo_n, 'nadeau_bengio_hi': hi_n, 'nadeau_bengio_p': p_n,
            'nadeau_bengio_excl_0': (lo_n > 0) or (hi_n < 0),
            'group_boot_diff': m_g, 'group_boot_lo': lo_g, 'group_boot_hi': hi_g,
            'group_boot_excl_0': (lo_g > 0) or (hi_g < 0),
        })
    return pd.DataFrame(rows).set_index('comparison')


comp_grouped = comparison_table(results_grp, preds_grp, groups, y, 'group-aware')
comp_random = comparison_table(results, preds, groups, y, 'random')
comp_grouped.to_csv(f'{OUTDIR}/paired_comparisons_grouped.csv')
comp_random.to_csv(f'{OUTDIR}/paired_comparisons_random.csv')

show = ['mean_diff', 'fold_boot_lo', 'fold_boot_hi', 'fold_boot_excl_0',
        'nadeau_bengio_lo', 'nadeau_bengio_hi', 'nadeau_bengio_excl_0',
        'group_boot_lo', 'group_boot_hi', 'group_boot_excl_0']
print('Paired differences in balanced accuracy -- GROUP-AWARE (primary)')
display(comp_grouped[show])
print('\nPaired differences in balanced accuracy -- RANDOM splits')
display(comp_random[show])

widen = ((comp_grouped['nadeau_bengio_hi'] - comp_grouped['nadeau_bengio_lo']) /
         (comp_grouped['fold_boot_hi'] - comp_grouped['fold_boot_lo']))
print(f'\nThe Nadeau-Bengio intervals are {widen.mean():.1f}x wider on average than the')
print('fold-level bootstrap. Any comparison that survives only under the narrow')
print('interval should not be claimed in the paper.')

Paired differences in balanced accuracy -- GROUP-AWARE (primary)


,mean_diff,fold_boot_lo,fold_boot_hi,fold_boot_excl_0,nadeau_bengio_lo,nadeau_bengio_hi,nadeau_bengio_excl_0,group_boot_lo,group_boot_hi,group_boot_excl_0
comparison,,,,,,,,,,
ZZFeatureMap - ZFeatureMap,-0.0452,-0.0703,-0.0207,True,-0.1740,0.0836,False,-0.0917,0.0052,False
ZZFeatureMap - Classical RBF,-0.0232,-0.0489,0.0015,False,-0.1534,0.1070,False,-0.0743,0.0283,False
ZZFeatureMap - Classical linear,-0.0475,-0.0735,-0.0222,True,-0.1823,0.0872,False,-0.1082,0.0063,False
ZZFeatureMap - Classical poly,-0.0477,-0.0714,-0.0252,True,-0.1689,0.0735,False,-0.0890,0.0019,False
ZFeatureMap - Classical RBF,0.0220,0.0089,0.0360,True,-0.0482,0.0922,False,0.0062,0.0340,True
ZFeatureMap - Classical linear,-0.0023,-0.0170,0.0118,False,-0.0785,0.0738,False,-0.0298,0.0128,False
ZFeatureMap - Classical poly,-0.0025,-0.0188,0.0137,False,-0.0881,0.0831,False,-0.0187,0.0156,False
ZZFeatureMap - Majority baseline,0.2818,0.2581,0.3042,True,0.1612,0.4024,True,0.2720,0.4311,True



Paired differences in balanced accuracy -- RANDOM splits


,mean_diff,fold_boot_lo,fold_boot_hi,fold_boot_excl_0,nadeau_bengio_lo,nadeau_bengio_hi,nadeau_bengio_excl_0,group_boot_lo,group_boot_hi,group_boot_excl_0
comparison,,,,,,,,,,
ZZFeatureMap - ZFeatureMap,-0.0246,-0.0417,-0.0081,True,-0.1109,0.0617,False,-0.0641,0.0104,False
ZZFeatureMap - Classical RBF,-0.0260,-0.0426,-0.0096,True,-0.1109,0.0589,False,-0.0682,0.0163,False
ZZFeatureMap - Classical linear,-0.0257,-0.0443,-0.0053,True,-0.1268,0.0753,False,-0.0712,0.0176,False
ZZFeatureMap - Classical poly,-0.0376,-0.0544,-0.0208,True,-0.1245,0.0494,False,-0.0711,-0.0100,True
ZFeatureMap - Classical RBF,-0.0014,-0.0109,0.0080,False,-0.0505,0.0477,False,-0.0178,0.0140,False
ZFeatureMap - Classical linear,-0.0011,-0.0155,0.0152,False,-0.0811,0.0788,False,-0.0233,0.0196,False
ZFeatureMap - Classical poly,-0.0130,-0.0267,-0.0002,True,-0.0812,0.0552,False,-0.0359,0.0090,False
ZZFeatureMap - Majority baseline,0.3073,0.2891,0.3257,True,0.2120,0.4026,True,0.2226,0.3791,True



The Nadeau-Bengio intervals are 5.2x wider on average than the
fold-level bootstrap. Any comparison that survives only under the narrow
interval should not be claimed in the paper.


## 12. Session record

Everything needed to reproduce the run, written next to the results.

In [43]:
manifest = {
    'qiskit_version': qiskit.__version__,
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'random_seed': RANDOM_SEED,
    'n_peptides': int(len(y)),
    'class_balance': np.bincount(y).tolist(),
    'n_groups': int(len(np.unique(groups))),
    'grouping_rule': {'shared_kmer_length': K_SHARED,
                      'identity_threshold': IDENTITY_THRESH,
                      'group_definition': 'connected components of the similarity graph (union-find)'},
    'cv_design': {'outer_splits': N_SPLITS_OUTER, 'outer_repeats': N_REPEATS_OUTER,
                  'inner_splits': N_SPLITS_INNER,
                  'inner_selection_metric': INNER_SELECTION_METRIC,
                  'representation_selected_in_inner_loop': True,
                  'selection_scope': 'per kernel family'},
    'preprocessing': 'MinMaxScaler(feature_range=(-pi, pi)) fitted on the training part '
                     'of each fold only, then multiplied by the bandwidth',
    'quantum_grid': PARAM_GRID_QUANTUM,
    'classical_grids': {k: {kk: [str(x) for x in vv] for kk, vv in v.items()}
                        for k, v in CLASSICAL_GRIDS.items()},
    'entanglement': ENTANGLEMENT,
    'kernel': 'exact statevector fidelity |<phi(x_i)|phi(x_j)>|^2, validated against '
              'qiskit Statevector.from_instruction to ~1e-15',
    'representations': {k: {'source': v, 'n_features': int(representations[k].shape[1])}
                        for k, v in REPRESENTATIONS.items()},
    'excluded_representations': {
        'moments_order1_z5.csv': 'duplicate of zscale_dataset.csv (mean z-scale, order 1)',
        'PC1_data_frame.csv': 'superseded by the 87-amino-acid PCA fit',
        'PCA_4_full_dataset_36.csv': 'dimension-matched control, belongs to the diagnostics notebook',
    },
}
json.dump(manifest, open(f'{OUTDIR}/run_manifest.json', 'w'), indent=2)
print(json.dumps(manifest, indent=2)[:1800], '...')
print(f'\nAll outputs written to ./{OUTDIR}/')
print('\n'.join('  ' + f for f in sorted(os.listdir(OUTDIR))))

{
  "qiskit_version": "1.4.5",
  "numpy_version": "1.26.4",
  "pandas_version": "2.3.2",
  "random_seed": 42,
  "n_peptides": 80,
  "class_balance": [
    42,
    38
  ],
  "n_groups": 43,
  "grouping_rule": {
    "shared_kmer_length": 5,
    "identity_threshold": 0.7777777777777778,
    "group_definition": "connected components of the similarity graph (union-find)"
  },
  "cv_design": {
    "outer_splits": 5,
    "outer_repeats": 20,
    "inner_splits": 4,
    "inner_selection_metric": "balanced_accuracy",
    "representation_selected_in_inner_loop": true,
    "selection_scope": "per kernel family"
  },
  "preprocessing": "MinMaxScaler(feature_range=(-pi, pi)) fitted on the training part of each fold only, then multiplied by the bandwidth",
  "quantum_grid": {
    "reps": [
      1,
      2
    ],
    "bandwidth": [
      0.05,
      0.1,
      0.2,
      0.5
    ],
    "C": [
      0.1,
      1,
      10
    ]
  },
  "classical_grids": {
    "linear": {
      "C": [
        "0.1",
  